# 누락 3개 작물 전처리 — AI Hub 다운로드 + Colab 처리

**처리 대상** (노지 04.애호박은 이미 정상 처리됨)
| 그룹 | 환경 | 작물 | 그룹 타입 |
|------|------|------|----------|
| facility_01_가지 | 시설 | 01.가지 | heldout |
| facility_02_고추 | 시설 | 02.고추 | train |
| outdoor_01_고추 | 노지 | 01.고추 | train |

**흐름**: AI Hub 다운로드 → 전처리 → zip → Drive 업로드 → 검증

**작물 1개씩** 다운로드 → 전처리 → 로컬 정리 순서로 진행 (Colab 디스크 절약)

---
## 1. Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 2. aihubshell 설치 + API 키 설정

In [ ]:
import os, subprocess

def _aihub_installed():
    try:
        r = subprocess.run(['aihubshell'], capture_output=True, text=True, timeout=5)
        return 'aihubshell version' in r.stdout
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return False

if _aihub_installed():
    print('✓ aihubshell 이미 설치되어 있음')
else:
    drive_bin = '/content/drive/MyDrive/aihubshell'
    if os.path.exists(drive_bin):
        os.system(f'cp "{drive_bin}" /usr/local/bin/aihubshell')
        os.system('chmod +x /usr/local/bin/aihubshell')
        if _aihub_installed():
            print('✓ aihubshell 설치 완료 (Drive 복사)')
        else:
            print('❌ 복사했지만 실행 실패')
            os.system('file /usr/local/bin/aihubshell')
    else:
        os.system('wget -q "https://aihub.or.kr/aihubdata/aihubshell" -O /usr/local/bin/aihubshell')
        os.system('chmod +x /usr/local/bin/aihubshell')
        r = subprocess.run(['file', '/usr/local/bin/aihubshell'], capture_output=True, text=True)
        if 'ELF' in r.stdout or 'shell script' in r.stdout:
            print('✓ aihubshell 설치 완료 (wget)')
        else:
            os.system('rm /usr/local/bin/aihubshell')
            print('⚠ Drive에 aihubshell 파일 업로드 후 재실행')

In [ ]:
# AI Hub API 키
API_KEY = '983A2DE9-4E27-4C4C-A704-1E51F116CF02'

# 다운로드 경로 설정
DATASET_FACILITY = 153   # 071. 시설 작물 질병 진단
DATASET_OUTDOOR  = 147   # 073. 노지 작물 질병 진단

RAW_DRIVE = '/content/drive/MyDrive/datasets'   # 원본 저장 위치 (config.py의 RAW_DATA_ROOT)

print(f'API_KEY: {API_KEY[:8]}...{API_KEY[-4:]}')
print(f'RAW_DRIVE: {RAW_DRIVE}')

In [ ]:
import subprocess, os
from pathlib import Path

def aihub_download(dataset_id: int, file_keys: str, label: str = ''):
    """
    AI Hub에서 fileSn 목록을 다운로드.
    dest = RAW_DATA_ROOT/raw/ 에 추출 → find_source_zips 구조 충족
    """
    dest = Path(str(RAW_DATA_ROOT)) / 'raw'
    dest.mkdir(parents=True, exist_ok=True)

    cmd = (
        f'cd "{dest}" && '
        f'aihubshell -mode d -datasetkey {dataset_id} '
        f'-filekey {file_keys} -aihubapikey {API_KEY}'
    )
    print(f'\n{"─"*60}')
    print(f'다운로드: {label}')
    print(f'  dataset={dataset_id}, filekey={file_keys}')
    print(f'  dest={dest}')
    ret = os.system(cmd)
    print('  ✓ 완료' if ret == 0 else f'  ⚠ 종료코드 {ret}')
    return ret


def aihub_list_files(dataset_id: int):
    result = subprocess.run(
        ['aihubshell', '-mode', 'l', '-datasetkey', str(dataset_id)],
        capture_output=True, text=True
    )
    print(result.stdout[:4000])
    return result.stdout


print('다운로드 헬퍼 준비 완료')

---
## 3. 헬퍼 모듈 로드 (Drive → sys.path)

In [ ]:
import sys

# Drive에 업로드한 preprocessing_code 폴더 경로
CODE_DIR = '/content/drive/MyDrive/Colab_PandaNum'
sys.path.insert(0, CODE_DIR)

from pathlib import Path
assert Path(CODE_DIR).exists(), f'❌ 경로 없음: {CODE_DIR}'
required = ['config.py', 'label_utils.py', 'crop_utils.py', 'split_utils.py', 'sampling_utils.py']
for fname in required:
    fpath = Path(CODE_DIR) / fname
    print(f'  {fname}: {"✓" if fpath.exists() else "✗ 없음!"}')
print('\n모듈 경로 추가 완료')

In [ ]:
!pip install opencv-python-headless scikit-learn tqdm -q

import csv, time, zipfile, shutil
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional

from config import (
    RAW_DATA_ROOT, LABEL_ZIP_PATH, OUTPUT_ROOT,
    ENV_EN, SUBFOLDER_TRAIN, SUBFOLDER_VAL,
    MAX_PER_CLASS, RISK_NAMES,
    TRAIN_GROUPS, HELDOUT_GROUPS,
    group_id, group_type_of,
)
from label_utils    import load_group_labels, summarize_labels
from split_utils    import assign_splits, split_distribution
from sampling_utils import apply_cap
from crop_utils     import preprocess_image
from tqdm.notebook  import tqdm

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('모듈 로드 완료')
print(f'RAW_DATA_ROOT : {RAW_DATA_ROOT}  ({"존재" if RAW_DATA_ROOT.exists() else "없음"})')
print(f'LABEL_ZIP_PATH: {LABEL_ZIP_PATH}  ({"존재" if LABEL_ZIP_PATH.exists() else "없음"})')
print(f'OUTPUT_ROOT   : {OUTPUT_ROOT}')

---
## 4. 업로드 위치 설정

In [ ]:
import config as _cfg

# Drive 경로: datasets/ 로 고정
# find_source_zips가 datasets/raw/071.*/ 경로를 탐색하려면 RAW_DATA_ROOT = datasets/ 이어야 함
# (aihub_download도 RAW_DATA_ROOT/raw/ 에 저장 → datasets/raw/ 로 일치)
RAW_DATA_ROOT = Path('/content/drive/MyDrive/datasets')
_cfg.RAW_DATA_ROOT = RAW_DATA_ROOT
print(f'RAW_DATA_ROOT → {RAW_DATA_ROOT}')

RAW_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# 업로드 위치 (전처리 결과 zip)
UPLOAD_DIR = Path('/content/drive/MyDrive/datasets/PandaNum/preprocessed')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
print(f'UPLOAD_DIR   : {UPLOAD_DIR}')

existing = sorted(UPLOAD_DIR.glob('*.zip'))
print(f'\n이미 있는 zip ({len(existing)}개):')
for z in existing:
    print(f'  {z.name} ({z.stat().st_size/(1024**2):,.0f} MB)')

In [ ]:
import shutil

# RAW_DATA_ROOT 바로 아래에 071./073. 폴더가 있으면 raw/ 안으로 이동
raw_dir = Path(str(RAW_DATA_ROOT)) / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)

for d in Path(str(RAW_DATA_ROOT)).iterdir():
    if d.is_dir() and (d.name.startswith('071.') or d.name.startswith('073.')):
        dst = raw_dir / d.name
        if not dst.exists():
            print(f'이동 중: {d.name} → raw/')
            shutil.move(str(d), str(dst))
            print(f'  ✓ 완료')
        else:
            print(f'이미 존재: raw/{d.name} (스킵)')

print('raw/ 구조 확인 완료')

---
## 5. 처리 함수 정의

In [ ]:
def find_source_zips(env: str, crop_folder: str) -> Dict[str, List[Path]]:
    result = {'train': [], 'val': []}
    if not RAW_DATA_ROOT.exists():
        return result
    env_code = '071' if env == '시설' else '073'
    for top_dir in RAW_DATA_ROOT.iterdir():
        if not top_dir.is_dir():
            continue
        for ds_dir in top_dir.iterdir():
            if not ds_dir.is_dir() or not ds_dir.name.startswith(f'{env_code}.'):
                continue
            data_dir = ds_dir / '01.데이터'
            if not data_dir.exists():
                continue
            for sub_aihub, key in [('1.Training', 'train'), ('2.Validation', 'val')]:
                crop_path = data_dir / sub_aihub / '원천데이터' / crop_folder
                if crop_path.exists():
                    result[key].extend(crop_path.glob('*.zip'))
    result['train'] = sorted(set(result['train']))
    result['val']   = sorted(set(result['val']))
    return result

In [ ]:
def process_group(env: str, crop_folder: str) -> Optional[Dict]:
    gtype = group_type_of(env, crop_folder)
    gid   = group_id(env, crop_folder)
    print('\n' + '='*70)
    print(f'▶ [{gtype}] {env} {crop_folder}  →  {gid}')
    print('='*70)
    t0 = time.time()

    print('\n[1/5] 라벨 로드 중...')
    labels = load_group_labels(LABEL_ZIP_PATH, env, crop_folder)
    if not labels:
        print('  ❌ 라벨 없음.')
        return None
    s = summarize_labels(labels)
    print(f"  → 총 {s['total']:,}장 (원본 {s['is_aug_false']:,} + 증강 {s['is_aug_true']:,})")

    print('\n[2/5] D-1 분할...')
    splits = assign_splits(labels, gtype)
    for split, info in split_distribution(splits).items():
        risks = {RISK_NAMES[k]: v for k, v in sorted(info['by_risk'].items())}
        print(f"  {split:18s}: {info['total']:>6,}장  {risks}")

    print(f'\n[3/5] 캡 {MAX_PER_CLASS} 적용...')
    capped = apply_cap(splits) if gtype == 'train_group' else splits
    for split, info in split_distribution(capped).items():
        risks = {RISK_NAMES[k]: v for k, v in sorted(info['by_risk'].items())}
        print(f"  {split:18s}: {info['total']:>6,}장  {risks}")

    print('\n[4/5] 원본 zip 탐색...')
    src_zips = find_source_zips(env, crop_folder)
    n_zips = sum(len(v) for v in src_zips.values())
    if n_zips == 0:
        print(f'  ❌ {RAW_DATA_ROOT} 에 zip 없음. 다운로드 먼저 확인.')
        return None
    for k, zips in src_zips.items():
        print(f'  {k}: {len(zips)}개')
        for z in zips:
            print(f'    - {z.name} ({z.stat().st_size/(1024**3):.2f} GB)')

    print('\n[5/5] 이미지 전처리 + 저장...')
    out_dir = OUTPUT_ROOT / gid
    img_dir = out_dir / 'images'
    img_dir.mkdir(parents=True, exist_ok=True)

    lookup = {l.image_filename: (l, s) for l, s in capped}
    manifest_rows = []
    counter = skipped_no_label = skipped_decode = 0

    for split_key, zip_list in src_zips.items():
        for zip_path in zip_list:
            with zipfile.ZipFile(zip_path, 'r') as zf:
                entries = [e for e in zf.infolist()
                           if Path(e.filename).name.lower().endswith(('.jpg', '.jpeg', '.png'))]
                for entry in tqdm(entries, desc=zip_path.name, leave=False):
                    name = Path(entry.filename).name
                    if name not in lookup:
                        skipped_no_label += 1
                        continue
                    info, split = lookup[name]
                    img_bytes = zf.open(entry).read()
                    if not img_bytes:                    # 빈 버퍼 방어 (facility 고추 이슈)
                        skipped_decode += 1
                        continue
                    processed = preprocess_image(img_bytes, info.bbox)
                    if processed is None:
                        skipped_decode += 1
                        continue
                    counter += 1
                    out_name = f'img_{counter:07d}.jpg'
                    with open(img_dir / out_name, 'wb') as f:
                        f.write(processed)
                    manifest_rows.append({
                        'file':        out_name,
                        'env':         info.env,
                        'crop_folder': info.crop_folder,
                        'crop_code':   info.crop_code,
                        'disease':     info.disease,
                        'risk':        info.risk,
                        'grow':        info.grow,
                        'original_id': info.original_id,
                        'is_aug':      info.is_aug,
                        'split':       split,
                        'group_type':  gtype,
                        'group_id':    gid,
                    })

    if manifest_rows:
        keys = list(manifest_rows[0].keys())
        with open(out_dir / 'manifest.csv', 'w', encoding='utf-8-sig', newline='') as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(manifest_rows)

    elapsed = time.time() - t0
    print(f'\n  ✓ {counter:,}장 저장  (스킵: 라벨없음 {skipped_no_label:,} / 디코드실패 {skipped_decode:,})')
    print(f'  소요: {elapsed/60:.1f}분')
    return {'group_id': gid, 'n_images': counter, 'elapsed_min': elapsed/60}

In [ ]:
def zip_and_upload(env: str, crop_folder: str) -> Optional[Path]:
    gid     = group_id(env, crop_folder)
    src     = OUTPUT_ROOT / gid
    zip_out = OUTPUT_ROOT / f'{gid}.zip'
    if not src.exists():
        print(f'❌ 전처리 결과 없음: {src}')
        return None

    # zip 압축
    print(f'\nzip 압축: {zip_out.name} ...', end='', flush=True)
    t0 = time.time()
    with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_STORED) as zf:
        for p in src.rglob('*'):
            if p.is_file():
                zf.write(p, p.relative_to(src.parent))
    size_mb = zip_out.stat().st_size / (1024**2)
    print(f' 완료 ({time.time()-t0:.1f}s, {size_mb:.1f} MB)')

    # Drive 업로드
    dst = UPLOAD_DIR / f'{gid}.zip'
    print(f'Drive 업로드: {dst.name} ...', end='', flush=True)
    t0 = time.time()
    shutil.copy(zip_out, dst)
    print(f' 완료 ({time.time()-t0:.1f}s)')
    return dst


def cleanup_local(env: str, crop_folder: str):
    gid     = group_id(env, crop_folder)
    src     = OUTPUT_ROOT / gid
    zip_out = OUTPUT_ROOT / f'{gid}.zip'
    for p in [src, zip_out]:
        if p.is_dir():
            shutil.rmtree(p); print(f'삭제: {p}')
        elif p.exists():
            p.unlink(); print(f'삭제: {p}')
    stat = os.statvfs('/content')
    print(f'Colab 디스크 여유: {stat.f_bavail * stat.f_frsize / (1024**3):.1f} GB')

print('함수 정의 완료')

In [ ]:
def verify_zip(env: str, crop_folder: str):
    """
    생성된 zip이 dataset.py 기대 구조인지 검증.
    - manifest.csv 존재 여부 확인
    - 쉼표 구분자 확인 (탭이면 ❌ 경고)
    - 컬럼, 행 수, split 분포 출력
    """
    import csv, io
    gid      = group_id(env, crop_folder)
    zip_path = UPLOAD_DIR / f'{gid}.zip'  # Drive에 업로드된 파일 기준

    print(f'\n{"="*60}')
    print(f'  zip 검증: {gid}')
    print(f'{"="*60}')

    if not zip_path.exists():
        print(f'  ❌ Drive에 zip 없음: {zip_path}')
        return

    with zipfile.ZipFile(zip_path, 'r') as zf:
        all_files  = zf.namelist()
        img_files  = [f for f in all_files if '/images/' in f and f.endswith('.jpg')]
        manifest_key = f'{gid}/manifest.csv'

        print(f'  이미지 수: {len(img_files):,}개')

        if manifest_key not in all_files:
            print(f'  ❌ manifest.csv 없음! (기대: {manifest_key})')
            print(f'     실제 파일 목록 (상위 5개): {all_files[:5]}')
            return

        raw  = zf.read(manifest_key)
        text = raw.decode('utf-8-sig')
        first_line = text.split('\n')[0]
        tab_cols   = len(first_line.split('\t'))
        comma_cols = len(first_line.split(','))

        if comma_cols >= 10:
            delim, delim_ok = ',', True
        elif tab_cols >= 10:
            delim, delim_ok = '\t', False
        else:
            delim, delim_ok = ',', True

        reader = csv.DictReader(io.StringIO(text), delimiter=delim)
        rows   = list(reader)
        from collections import Counter
        split_cnt = Counter(r.get('split','?') for r in rows)

        status = '✅ 정상' if delim_ok else '❌ 탭 구분자 — merge 오류 발생!'
        print(f'  구분자: {"쉼표" if delim_ok else "탭"} → {status}')
        print(f'  컬럼({len(reader.fieldnames)}개): {reader.fieldnames}')
        print(f'  총 행 수: {len(rows):,}행')
        print(f'  split 분포: {dict(sorted(split_cnt.items()))}')
        if rows:
            r = rows[0]
            print(f'  첫 번째 행: file={r.get("file")} | split={r.get("split")} | risk={r.get("risk")}')

        if not delim_ok:
            print('\n  ❌ proc 셀을 다시 실행해서 pipeline으로 재생성하세요.')
        else:
            print('\n  ✅ Drive 업로드 완료 — RunPod에서 다운로드하세요.')

print('verify_zip 함수 정의 완료')

---
## 6. 시설 01.가지 (held-out)

> ✅ Drive에 Training 원본 완전 (0.정상_(1~3), 1.질병, 9.증강 총 5개 파일)  
> ✅ Validation 원본 완전 (0.정상 772MB; 1.질병·9.증강 빈파일은 AI Hub 원래 이슈)  
> **dl-gaji-train / dl-gaji-val 셀 스킵 가능 — proc-gaji 만 실행하면 됨**

In [ ]:
# 시설 01.가지 Training 원천데이터 (전체)
# 0.정상_(1) 28GB | 0.정상_(2) 18GB | 0.정상_(3) 14GB | 1.질병 1GB | 9.증강 19GB
# 합계 약 80 GB
aihub_download(
    dataset_id = DATASET_FACILITY,
    file_keys  = '46844,46845,46846,46847,46848',
    label      = '시설 01.가지 Training 원천데이터 전체 (~80 GB)',
)

In [ ]:
# 시설 01.가지 Validation 원천데이터 (전체)
# 0.정상 772MB | 1.질병 22B(빈파일) | 9.증강 22B(빈파일)
aihub_download(
    dataset_id = DATASET_FACILITY,
    file_keys  = '46741,46742,46743',
    label      = '시설 01.가지 Validation 원천데이터 전체 (~772 MB)',
)

In [ ]:
env, crop = '시설', '01.가지'
process_group(env, crop)
zip_and_upload(env, crop)
cleanup_local(env, crop)
verify_zip(env, crop)

---
## 7. 시설 02.고추 (train)

> ⚠️ Training: `9.증강_(2).zip` 이 1 GB (18 GB 필요) — **dl-gochu-fac-train 실행 필수**  
> ✅ Validation: 완전 — dl-gochu-fac-val 스킵 가능  
> 빈 버퍼 방어 코드 포함되어 있음

In [ ]:
# 시설 02.고추 Training 원천데이터 (전체)
# 0.정상 10GB | 1.질병 4GB | 9.증강_(1) 20GB | 9.증강_(2) 18GB | 9.증강_(3) 23GB
# 합계 약 75 GB
aihub_download(
    dataset_id = DATASET_FACILITY,
    file_keys  = '46849,46850,46851,46852,46853',
    label      = '시설 02.고추 Training 원천데이터 전체 (~75 GB)',
)

In [ ]:
# 시설 02.고추 Validation 원천데이터 (전체)
# 0.정상 49MB | 1.질병 1GB | 9.증강 21GB
# 합계 약 22 GB
aihub_download(
    dataset_id = DATASET_FACILITY,
    file_keys  = '46744,46745,46746',
    label      = '시설 02.고추 Validation 원천데이터 전체 (~22 GB)',
)

In [ ]:
env, crop = '시설', '02.고추'
process_group(env, crop)
zip_and_upload(env, crop)
cleanup_local(env, crop)
verify_zip(env, crop)

---
## 8. 노지 01.고추 (train)

> ❌ Training: `0.정상`, `1.질병` 없음 + `9.증강_(1)` 2 GB (29 GB 필요) — **dl-gochu-out-train 실행 필수**  
> ✅ Validation: 완전 — dl-gochu-out-val 스킵 가능

In [ ]:
# 노지 01.고추 Training 원천데이터 (전체)
# 0.정상 11GB | 1.질병 3GB | 9.증강_(1) 29GB | 9.증강_(2) 26GB
# 합계 약 69 GB
aihub_download(
    dataset_id = DATASET_OUTDOOR,
    file_keys  = '46445,46446,46447,46448',
    label      = '노지 01.고추 Training 원천데이터 전체 (~69 GB)',
)

In [ ]:
# 노지 01.고추 Validation 원천데이터 (전체)
# 0.정상 1GB | 1.질병 422MB | 9.증강 7GB
# 합계 약 8 GB
aihub_download(
    dataset_id = DATASET_OUTDOOR,
    file_keys  = '46417,46418,46419',
    label      = '노지 01.고추 Validation 원천데이터 전체 (~8 GB)',
)

In [ ]:
env, crop = '노지', '01.고추'
process_group(env, crop)
zip_and_upload(env, crop)
cleanup_local(env, crop)
verify_zip(env, crop)

---
## 9. 노지 04.애호박 (train) — ✅ 이미 완료

> 노지 04.애호박은 학습 결과 샘플러에서 정상 확인됨 (outdoor_04_애호박 포함).  
> 아래 셀은 실행하지 마세요.

In [ ]:
# 노지 04.애호박 Training 원천데이터 (전체)
# 0.정상 16GB | 1.질병 2GB | 9.증강_(1) 29GB | 9.증강_(2) 21GB
# 합계 약 68 GB
aihub_download(
    dataset_id = DATASET_OUTDOOR,
    file_keys  = '46456,46457,46458,46459',
    label      = '노지 04.애호박 Training 원천데이터 전체 (~68 GB)',
)

In [ ]:
# 노지 04.애호박 Validation 원천데이터 (전체)
# 0.정상 2GB | 1.질병 280MB | 9.증강 6GB
# 합계 약 8 GB
aihub_download(
    dataset_id = DATASET_OUTDOOR,
    file_keys  = '46426,46427,46428',
    label      = '노지 04.애호박 Validation 원천데이터 전체 (~8 GB)',
)

In [ ]:
# 전처리 + zip + Drive 업로드 + 로컬 정리
env, crop = '노지', '04.애호박'
process_group(env, crop)
zip_and_upload(env, crop)
cleanup_local(env, crop)

---
## 10. 완료 확인

In [ ]:
print('=== Drive 업로드 완료 목록 ===')
zips = sorted(UPLOAD_DIR.glob('*.zip'))
total_gb = sum(z.stat().st_size for z in zips) / (1024**3)
for z in zips:
    size_mb = z.stat().st_size / (1024**2)
    print(f'  {z.name:40s} {size_mb:>8,.1f} MB')
print(f'\n합계: {len(zips)}개, {total_gb:.2f} GB')

# 이번 배치 3개 확인
required = [
    'facility_01_가지.zip',
    'facility_02_고추.zip',
    'outdoor_01_고추.zip',
]
print('\n[이번 배치 3개 확인]')
all_ok = True
for r in required:
    exists = (UPLOAD_DIR / r).exists()
    print(f'  {r:40s} {"✅" if exists else "❌ 없음"}')
    if not exists:
        all_ok = False

print()
if all_ok:
    print('✅ 3개 모두 Drive 업로드 완료 — RunPod에서 다운로드 후 merge_manifests 재실행')
else:
    print('❌ 누락 zip 있음 — 해당 작물 proc 셀 재실행 필요')

# 검증도 한 번에
print('\n[zip 구조 재검증]')
for env, crop in [('시설', '01.가지'), ('시설', '02.고추'), ('노지', '01.고추')]:
    verify_zip(env, crop)